# 1. Countdown Puzzle: AST Verifier & Dataset Generator
**HLT 2026 - Reasoning with Verifiable Rewards**

This notebook sets up the deterministic environment for the **Countdown** arithmetic game:
1. **Deterministic AST-based Verifier** to securely and strictly evaluate model-generated equations.
2. **Solvable Puzzle Generator** ensuring 100% of generated instances have valid solutions.
3. **Unit Test Suite** to guarantee robustness against reward hacking and parsing exploits.

> 💡 **Compute Note**: This notebook **does not require a GPU** and runs smoothly on standard CPU (local or Google Colab).

## 1. Setup & Imports

In [ ]:
import os
import ast
import re
import json
import random
import itertools
from typing import List, Dict, Tuple, Optional, Any

## 2. Deterministic AST Verifier

The verifier enforces:
1. **Tag Extraction**: Captures reasoning within `<think>...</think>` and final equation in `<answer>...</answer>`.
2. **Syntax & Security Whitelist**: Employs an Abstract Syntax Tree (AST) to allow only basic arithmetic operations (`+`, `-`, `*`, `/`, parentheses, and numbers).
3. **Number Constraints**: Each number used must be present in the initial set and used **at most once**.
4. **Target Equality**: The computed result must strictly match the target integer.

In [ ]:
class CountdownVerifier:
    """
    Deterministic and secure AST-based verifier for the Countdown arithmetic game.
    """
    ALLOWED_OPERATORS = {
        ast.Add: lambda a, b: a + b,
        ast.Sub: lambda a, b: a - b,
        ast.Mult: lambda a, b: a * b,
        ast.Div: lambda a, b: a / b if b != 0 else None,
        ast.FloorDiv: lambda a, b: a // b if b != 0 else None,
        ast.USub: lambda a: -a,
        ast.UAdd: lambda a: +a
    }

    # Explicitly include operator node classes because ast.walk visits them as children of BinOp/UnaryOp
    ALLOWED_NODE_TYPES = (
        ast.Expression,
        ast.BinOp,
        ast.UnaryOp,
        ast.Constant,
        ast.Add,
        ast.Sub,
        ast.Mult,
        ast.Div,
        ast.FloorDiv,
        ast.USub,
        ast.UAdd
    )

    @staticmethod
    def extract_tags(text: str) -> Tuple[Optional[str], Optional[str]]:
        """Extract reasoning and final equation from XML-like tags <think> and <answer>."""
        think_match = re.search(r"<think>(.*?)</think>", text, re.DOTALL | re.IGNORECASE)
        thought = think_match.group(1).strip() if think_match else None

        answer_match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
        answer = answer_match.group(1).strip() if answer_match else None

        if not answer:
            answer_partial = re.search(r"<answer>(.*)", text, re.DOTALL | re.IGNORECASE)
            if answer_partial:
                answer = answer_partial.group(1).strip().split("\n")[0].strip()

        return thought, answer

    @classmethod
    def is_safe_ast(cls, tree: ast.AST) -> bool:
        """Verify that the AST contains ONLY allowed arithmetic nodes and operators."""
        for node in ast.walk(tree):
            if not isinstance(node, cls.ALLOWED_NODE_TYPES):
                return False
            if isinstance(node, ast.BinOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.UnaryOp) and type(node.op) not in cls.ALLOWED_OPERATORS:
                return False
            if isinstance(node, ast.Constant):
                if isinstance(node.value, bool) or not isinstance(node.value, (int, float)):
                    return False
        return True

    @classmethod
    def evaluate_ast(cls, node: ast.AST) -> Optional[float]:
        """Recursively evaluate an AST node, allowing only basic arithmetic operations."""
        if isinstance(node, ast.Expression):
            return cls.evaluate_ast(node.body)

        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
            return float(node.value)

        elif isinstance(node, ast.UnaryOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            val = cls.evaluate_ast(node.operand)
            return op_func(val) if val is not None else None

        elif isinstance(node, ast.BinOp):
            op_func = cls.ALLOWED_OPERATORS.get(type(node.op))
            if not op_func:
                return None
            left_val = cls.evaluate_ast(node.left)
            right_val = cls.evaluate_ast(node.right)
            if left_val is None or right_val is None:
                return None
            try:
                res = op_func(left_val, right_val)
                return res
            except (ZeroDivisionError, OverflowError):
                return None

        return None

    @classmethod
    def extract_numbers_from_ast(cls, node: ast.AST) -> List[int]:
        """Extract all numbers used in the parsed AST expression."""
        numbers = []
        for n in ast.walk(node):
            if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
                if float(n.value).is_integer():
                    numbers.append(int(n.value))
                else:
                    numbers.append(n.value)
        return numbers

    @classmethod
    def verify(cls, available_numbers: List[int], target: int, completion: str) -> Dict[str, Any]:
        """
        Full verification of the model completion.
        Returns a dictionary containing formatting, validity, and correctness status.
        """
        thought, answer_str = cls.extract_tags(completion)

        result = {
            "has_format": bool(thought is not None and answer_str is not None),
            "thought": thought,
            "answer_str": answer_str,
            "valid_syntax": False,
            "valid_numbers": False,
            "used_numbers": [],
            "computed_value": None,
            "is_correct": False,
            "error_msg": None
        }

        if not answer_str:
            result["error_msg"] = "Tag <answer> not found or empty"
            return result

        # 1. Parse AST and enforce security whitelist
        try:
            clean_expr = answer_str.split("=")[0].strip()
            tree = ast.parse(clean_expr, mode='eval')
            if not cls.is_safe_ast(tree):
                result["error_msg"] = "Expression contains unallowed elements (only +, -, *, /, parentheses, and numbers)"
                return result
            result["valid_syntax"] = True
        except Exception as e:
            result["error_msg"] = f"AST syntax error: {e}"
            return result

        # 2. Check number usage constraints
        used_numbers = cls.extract_numbers_from_ast(tree)
        result["used_numbers"] = used_numbers

        available_copy = list(available_numbers)
        valid_nums = True
        for num in used_numbers:
            if num in available_copy:
                available_copy.remove(num)
            else:
                valid_nums = False
                break

        result["valid_numbers"] = valid_nums and (len(used_numbers) > 0)
        if not result["valid_numbers"]:
            result["error_msg"] = f"Invalid numbers or numbers used more than once: {used_numbers} (Available: {available_numbers})"
            return result

        # 3. Evaluate expression
        computed = cls.evaluate_ast(tree)
        result["computed_value"] = computed

        if computed is None:
            result["error_msg"] = "Evaluation failed (e.g. division by zero)"
            return result

        # 4. Check target equality (with float tolerance for exact divisions)
        if abs(computed - target) < 1e-6:
            result["is_correct"] = True
        else:
            result["error_msg"] = f"Result {computed} != target {target}"

        return result

## 3. Verifier Unit Tests
Run a battery of unit tests covering valid cases, number reuse, hallucinated numbers, divide-by-zero, and arbitrary code injection:

In [ ]:
def test_verifier():
    nums = [84, 83, 34, 72]
    target = 39

    # Test 1: Correct equation with valid formatting
    out1 = "<think>Solve: (84 + 72) - (83 + 34) = 156 - 117 = 39</think><answer>(84 + 72) - (83 + 34)</answer>"
    res1 = CountdownVerifier.verify(nums, target, out1)
    assert res1["is_correct"] and res1["has_format"], f"Failed test 1: {res1}"

    # Test 2: Illegal number reuse (84 used twice)
    out2 = "<think>Try</think><answer>84 + 84 - 72</answer>"
    res2 = CountdownVerifier.verify(nums, target, out2)
    assert not res2["valid_numbers"], "Failed test 2: unallowed number reuse accepted"

    # Test 3: Number not present in the available list (10)
    out3 = "<think>Calculate</think><answer>10 + 29</answer>"
    res3 = CountdownVerifier.verify(nums, target, out3)
    assert not res3["valid_numbers"], "Failed test 3: hallucinated number accepted"

    # Test 4: Division by zero
    out4 = "<think>...</think><answer>84 / (34 - 34)</answer>"
    res4 = CountdownVerifier.verify(nums, target, out4)
    assert not res4["is_correct"], "Failed test 4: division by zero unhandled"

    # Test 5: Malicious / non-arithmetic syntax
    out5 = "<think>...</think><answer>__import__('os').system('ls')</answer>"
    res5 = CountdownVerifier.verify(nums, target, out5)
    assert not res5["valid_syntax"], "Failed test 5: non-arithmetic code accepted"

    print("✅ All 5 verifier unit tests passed successfully!")

test_verifier()

## 4. Solvable Puzzle Generator

We implement an exhaustive recursive solver that finds all reachable positive integers for 4 given numbers and filters out trivial solutions:

In [ ]:
def solve_countdown_all(numbers: List[int]) -> Dict[int, str]:
    """Find all reachable positive integer targets for the given list of numbers."""
    targets = {}

    for length in range(2, len(numbers) + 1):
        for subset in itertools.permutations(numbers, length):
            def eval_tree(items):
                if len(items) == 1:
                    return [(items[0], str(items[0]))]
                results = []
                for split in range(1, len(items)):
                    lefts = eval_tree(items[:split])
                    rights = eval_tree(items[split:])
                    for lv, le in lefts:
                        for rv, re in rights:
                            results.append((lv + rv, f"({le} + {re})"))
                            if lv - rv > 0:
                                results.append((lv - rv, f"({le} - {re})"))
                            results.append((lv * rv, f"({le} * {re})"))
                            if rv > 0 and lv % rv == 0:
                                results.append((lv // rv, f"({le} / {re})"))
                return results

            for v, e in eval_tree(list(subset)):
                if 1 <= v <= 200:
                    targets[v] = e

    return targets

def generate_countdown_dataset(num_samples: int, seed: int = 42) -> List[Dict[str, Any]]:
    """Generate a dataset of Countdown puzzles with guaranteed solvability."""
    random.seed(seed)
    dataset = []
    seen = set()

    attempts = 0
    while len(dataset) < num_samples and attempts < num_samples * 25:
        attempts += 1
        numbers = [random.randint(1, 100) for _ in range(4)]
        key = tuple(sorted(numbers))
        if key in seen:
            continue

        reachable = solve_countdown_all(numbers)
        # Ensure non-trivial target (not equal to any starting number and >= 10)
        valid_targets = [t for t in reachable.keys() if t not in numbers and t >= 10]

        if valid_targets:
            chosen_target = random.choice(valid_targets)
            sample = {
                "id": len(dataset) + 1,
                "numbers": numbers,
                "target": chosen_target,
                "example_solution": reachable[chosen_target],
                "prompt": f"Using the numbers {numbers}, create an equation that equals {chosen_target}. You can use basic arithmetic operations (+, -, *, /) and each number can be used at most once. Show your reasoning in <think>...</think> and place the final equation in <answer>...</answer>."
            }
            dataset.append(sample)
            seen.add(key)

    return dataset

## 5. Dataset Generation & Export (Train & Test Sets)

In [ ]:
# Automatically set data directory whether running locally or in Colab
data_dir = "data" if not os.path.basename(os.getcwd()) == "notebooks" else "../data"
os.makedirs(data_dir, exist_ok=True)

print("Generating Test Dataset (200 samples)...")
test_data = generate_countdown_dataset(200, seed=123)

print("Generating Train Dataset (1000 samples)...")
train_data = generate_countdown_dataset(1000, seed=456)

test_file = os.path.join(data_dir, "test_countdown.jsonl")
train_file = os.path.join(data_dir, "train_countdown.jsonl")

with open(test_file, "w", encoding="utf-8") as f:
    for item in test_data:
        f.write(json.dumps(item) + "\n")

with open(train_file, "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

print(f"Saved: {len(train_data)} train samples to {train_file}")
print(f"Saved: {len(test_data)} test samples to {test_file}")
print("\nSample problem:")
print(json.dumps(test_data[0], indent=2))